In [ ]:
# ---- Configuration -------------------------------------------------
# Set these to match your environment, then run the cells below.
# On Colab, mount Drive first and point BASE at your project folder.

BASE      = '/content/drive/MyDrive/final'                # project root
SUPERGLUE = '/content/drive/MyDrive/CVProject/superglue'   # SuperGlue checkout
VIDEO     = f'{BASE}/1.mp4'                                # input dashcam video

FRAME_H, FRAME_W     = 720, 1280   # source frame size
TOPVIEW_W, TOPVIEW_H = 556, 483    # warped top-view canvas size
# ---------------------------------------------------------------------


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# Tensorflow
import tensorflow as tf
print(tf.__version__)

# I/O libraries
import os
from io import BytesIO
import tarfile
import tempfile
from six.moves import urllib

# Helper libraries
import matplotlib
from matplotlib import gridspec
from matplotlib import pyplot as plt
import numpy as np
from PIL import Image
import cv2 as cv
from google.colab.patches import cv2_imshow
import cv2
from tqdm import tqdm
import IPython
from sklearn.metrics import confusion_matrix
from tabulate import tabulate

# Comment this out if you want to see Deprecation warnings
import warnings
warnings.simplefilter("ignore", DeprecationWarning)

In [ ]:
# # Replace the URL with the GitHub repository URL you want to clone
# repo_url = "https://github.com/bimalka98/Stitch-images-using-SuperGlue-GNN"

# # Replace the destination path with your desired destination in Google Drive
# destination_path = "{SUPERGLUE}"

# # Clone the repository
# !git clone $repo_url $destination_path


In [ ]:
import cv2
import os

# Set your video file name
video_name = '{BASE}/1.mp4'

# Set the output directory for frames
output_dir = '{BASE}/out_frames/'
output_dir1 = '{BASE}/HybridNets/demo/img/'
counter = 0

# Create output directory if it doesn't exist

os.makedirs(output_dir)
#if not os.path.exists(output_dir1):
os.makedirs(output_dir1)

# Open video capture
cap = cv2.VideoCapture(video_name)

# Get video properties
num_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

# Loop through frames
for frame_num in range(0, num_frames,5):
    print(frame_num)

    # Set the video capture to the desired frame
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_num)

    # Read the frame
    ret, frame = cap.read()

    if not ret:
        break

    # Save the frame with the desired name format
    frame = cv2.resize(frame, (1280, 720))
    frame_filename = f'{output_dir}road-{counter:02d}.jpg'
    cv2.imwrite(frame_filename, frame)
    frame = cv2.resize(frame, (1280, 720))
    frame_filename = f'{output_dir1}{counter}.jpg'
    cv2.imwrite(frame_filename, frame)
    counter += 1

# Release video capture
cap.release()


In [ ]:
%cd {BASE}/

In [ ]:

import os
counter=0
folder_path = 'out_frames/'  # Change this to the path of your folder

# Ensure the folder path is valid
if os.path.exists(folder_path):
    # Get the list of files in the folder
    files = [f for f in os.listdir(folder_path) if os.path.isfile(os.path.join(folder_path, f))]

    # Print the number of files
    counter=len(files)
    print(len(files))


In [ ]:
# generating the necessary txt file to input for the super glue algorithm
img_name = 'road' # set of outdoor images
num_images = counter
# Order of the images. To stitch left and right images as depicted in the below
order = range(1,num_images -1)
with open('output_frames.txt', 'w') as file:
    for i in order:
        file.write("{img}-{:02}.jpg {img}-{:02}.jpg\n".format(i,i-1, img = img_name))

In [ ]:
!python {SUPERGLUE}/match_pairs.py  --resize -1 \
                        --superglue outdoor \
                        --max_keypoints 2048 \
                        --nms_radius 5 \
                        --resize_float \
                        --input_dir out_frames/ \
                        --input_pairs output_frames.txt \
                        --output_dir road_panorama/output \
                        --viz \
                        --keypoint_threshold 0.05 \
                        --match_threshold 0.9

In [ ]:
#generatig the npz files for extract matching information
npz_files = ["{img}-{:02}_{img}-{:02}_matches.npz".format(i,i-1, img = img_name) for i in order]
for file in npz_files:
    path = 'road_panorama/output/'+file
    npz = np.load(path)
print(npz.files)

In [ ]:
print(npz_files)

In [ ]:
# extracting information from the npz files
def loadNPZ(npz_file):
    npz = np.load('road_panorama/output/'+ npz_file)
    point_set1 = npz['keypoints0'][npz['matches']>-1]
    matching_indexes =  npz['matches'][npz['matches']>-1] # -1 if the keypoint is unmatched
    point_set2 = npz['keypoints1'][matching_indexes]
    print("Number of matching points for the findHomography algorithm:")
    print("In left  image:", len(point_set1),"\nIn right image:", len(point_set2))
    return point_set1, point_set2

In [ ]:
def pltSourceImages(imageSet):
# Load images
    im_left = cv2.imread(f'{BASE}/out_frames/road-{imageSet:02d}.jpg')
    im_right = cv2.imread(f'{BASE}/out_frames/road-{imageSet-1:02d}.jpg')

    # Check if images are loaded successfully
    if im_left is None or im_right is None:
        print(f"Error: Unable to load images for imageSet {imageSet}.")
        return

    # Convert BGR to RGB
    im_left_rgb = cv2.cvtColor(im_left, cv2.COLOR_BGR2RGB)
    im_right_rgb = cv2.cvtColor(im_right, cv2.COLOR_BGR2RGB)

    # Load masks
    mask_1 = cv2.imread(f"{BASE}/HybridNets/demo_result2/{imageSet}.jpg", cv2.IMREAD_GRAYSCALE)
    white_coordinates_1 = np.column_stack(np.nonzero(mask_1))

    mask_2 = cv2.imread(f"{BASE}/HybridNets/demo_result2/{imageSet-1}.jpg", cv2.IMREAD_GRAYSCALE)
    white_coordinates_2 = np.column_stack(np.nonzero(mask_2))  # Corrected from mask_1 to mask_2


    wc1=[]
    for point in white_coordinates_1.astype(np.int32):
        p=(point[1],point[0])
        wc1.append(p)
    wc2=[]
    for point in white_coordinates_2.astype(np.int32):
        p=(point[1],point[0])
        wc2.append(p)

    # Find common coordinates efficiently using set intersection
    coordinates_set_1 = set(map(tuple, wc1))
    p1 = set(map(tuple, point_set1))
    coordinates_set_2 = set(map(tuple, wc2))
    p2 = set(map(tuple, point_set2))

    new_set1 = coordinates_set_1.intersection(p1)

    n1=[]
    for x in new_set1:
      n1.append(list(x))
    print(n1)


    idx=[]

    print("IDX")

    for i in range(0,len(point_set1)):
      for j in range(0,len(n1)):
        if np.array_equal(point_set1[i], n1[j]):
          idx.append(i)
    print(idx)



    p1_list = list(p1)
    p2_list = list(p2)
    new_list=list(new_set1)
    new_point1=[]
    new_point2=[]

    for i in idx:


      new_point1.append(point_set1[i])

      new_point2.append(point_set2[i])






    new_point1 = np.array(new_point1)

    new_point2 = np.array(new_point2)




    for point in new_point1.astype(np.int32):
        cv2.circle(im_left_rgb, tuple(point), radius=8, color=(255, 255, 0), thickness=-1)

    for point in new_point2.astype(np.int32):
        cv2.circle(im_right_rgb, tuple(point), radius=8, color=(255, 255, 0), thickness=-1)

    # Display the images
    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    axes[0].imshow(im_left_rgb)
    axes[0].set_title('Left Image')

    axes[1].imshow(im_right_rgb)
    axes[1].set_title('Right Image')

    plt.show()
    # print("new_point1")
    # print(new_point1)
    # print("new_point2")
    # print(new_point2)
    return new_point1,new_point2

In [ ]:
def plotMatches(imageSet):
    print("XX")
    plt.figure(figsize=(10,10))
    matched_points = cv.imread('road_panorama/output/road-{:02}_road-{:02}_matches.png'.\
                     format(imageSet, imageSet -1),cv.IMREAD_ANYCOLOR)
    plt.imshow(matched_points, cmap='gray', vmin = 0, vmax = 255)
    plt.show()

In [ ]:
# h1t=np.array(
# [[-6.50846679e-02 ,-7.77000898e-01  ,2.86187277e+02],
#  [-5.40786908e-02 ,-1.28473406e+00  ,4.68234614e+02],
#  [-1.42152039e-04 ,-2.91360877e-03  ,1.00000000e+00]]
# )

In [ ]:
h1t=np.array([[-4.08101432e-02, -5.16599153e-01,  2.70619659e+02],
 [-2.83020592e-02 ,-6.85416529e-01  ,3.54386187e+02],
 [-9.73128797e-05 ,-2.00980612e-03 , 1.00000000e+00]])

In [ ]:
# ---- Per-frame top-view projection (CORRECTED) ----------------------
# Changes from the original submission:
#   1. The homography now accumulates across frames instead of being rebuilt
#      from h1t each iteration. See src/topview.py :: HomographyChain.
#   2. findHomography uses RANSAC so outlier matches cannot skew the fit.
#   3. Frames that fail estimation are skipped without breaking the chain.

import sys
sys.path.insert(0, '../src')

import cv2
import numpy as np
import ast
import os
from topview import (
    HomographyChain, estimate_pairwise_homography,
    project_points, warp_to_topview,
)

chain = HomographyChain(h1t)
homographies = {}

for imgSet in range(6, 70):
    ind = imgSet + 1

    point_set1, point_set2 = loadNPZ(npz_files[imgSet])
    p1, p2 = pltSourceImages(ind)

    H, status = estimate_pairwise_homography(p2, p1)
    if H is None:
        print(f"frame {ind}: too few matches, skipping")
        continue

    # Chain onto the running product rather than restarting from h1t
    hom = chain.advance(H)
    homographies[imgSet] = hom

    inliers = int(status.sum()) if status is not None else 0
    print(f"frame {ind}: {len(p1)} matches, {inliers} inliers")

    frame = cv2.imread(f'out_frames/road-{ind:02}.jpg', cv2.IMREAD_ANYCOLOR)
    mask = cv2.imread(f'{BASE}/HybridNets/demo_result2/{ind}.jpg',
                      cv2.IMREAD_GRAYSCALE)

    result = warp_to_topview(frame, mask, hom, (TOPVIEW_W, TOPVIEW_H))

    # Project detected vehicle contact points into the same plane
    midpoint_file = f"{BASE}/HybridNets/midpoints2/midpoints_image_{imgSet}.txt"
    if os.path.exists(midpoint_file):
        with open(midpoint_file) as f:
            coordinates = [ast.literal_eval(line.strip()) for line in f]

        for coord in coordinates:
            x_sat, y_sat = project_points(np.array([coord[:2]]), hom)[0]
            cv2.circle(result, (int(x_sat), int(y_sat)), 5, (0, 255, 0), -1)

    output_folder = f'{BASE}/top/'
    os.makedirs(output_folder, exist_ok=True)
    cv2.imwrite(os.path.join(output_folder, f'top-{imgSet:02}.png'), result)

print(f"\nprojected {len(homographies)} frames")



In [ ]:
%cd {BASE}/top/

In [ ]:
# ---- Panorama composition (CORRECTED) -------------------------------
# The original used hardcoded paste offsets (y_offset -= 100, x_offset += 50)
# and index-range branches, because the homographies were not accumulating
# and so could not position frames themselves.
#
# With a correct chain, placement follows from the transforms: compute the
# bounding box over all warped frames, translate it to the origin, and warp
# each frame directly into the shared canvas.

import sys
sys.path.insert(0, '../src')

import cv2
import numpy as np
from topview import topview_bounds, translation_to_origin, compose_panorama

frame_shape = (FRAME_H, FRAME_W)
Hs = [homographies[k] for k in sorted(homographies)]

x_min, y_min, x_max, y_max = topview_bounds(frame_shape, Hs)
T = translation_to_origin(x_min, y_min)

canvas_w = int(np.ceil(x_max - x_min))
canvas_h = int(np.ceil(y_max - y_min))
print(f"canvas: {canvas_w} x {canvas_h}")

warped = []
for k in sorted(homographies):
    img = cv2.imread(f'{BASE}/top/top-{k:02}.png', cv2.IMREAD_COLOR)
    if img is None:
        continue
    warped.append(cv2.warpPerspective(img, T, (canvas_w, canvas_h)))

panorama = compose_panorama(warped, (canvas_w, canvas_h))
cv2.imwrite(f'{BASE}/panorama.png', panorama)
cv2_imshow(panorama)



In [ ]:
# %cd {SUPERGLUE}/crop/HybridNets
# !python hybridnets_test_videos.py -w weights/hybridnets.pth --source demo/video --output demo_result

In [ ]:
!git clone https://github.com/datvuthanh/HybridNets

!pip install -r requirements.txt

In [ ]:
!pip install requests


In [ ]:
%cd {BASE}/HybridNets

In [ ]:
import os
import requests

url = "https://github.com/datvuthanh/HybridNets/releases/download/v1.0/hybridnets.pth"
destination_path = "weights/hybridnets.pth"

# Create the directory if it doesn't exist
os.makedirs(os.path.dirname(destination_path), exist_ok=True)

response = requests.get(url, stream=True)
with open(destination_path, "wb") as file:
    for chunk in response.iter_content(chunk_size=128):
        file.write(chunk)


In [ ]:
!pip install timm==0.6.13


In [ ]:
!pip install pretrainedmodels


In [ ]:
!pip install efficientnet_pytorch


In [ ]:
# Download end-to-end weights
#curl --create-dirs -L -o weights/hybridnets.pth https://github.com/datvuthanh/HybridNets/releases/download/v1.0/hybridnets.pth

# Image inference
!python hybridnets_test.py -w weights/hybridnets.pth --source {BASE}/HybridNets/demo/img --output demo_result2 --imshow False --imwrite True

# Video inference
#!python hybridnets_test_videos.py -w weights/hybridnets.pth --source demo/video --output demo_result

# Result is saved in a new folder called demo_result

In [ ]:
import os

current_directory = os.getcwd()
print("Current Directory:", current_directory)


In [ ]:
!python hybridnets_test_videos.py -w weights/hybridnets.pth --source {BASE} --output demo_result1

In [ ]:
!pip install imread_from_url


In [ ]:
!pip install onnxruntime


In [ ]:


cap = cv2.VideoCapture('{BASE}/1.mp4')

# Read the first frame
for i in range(0,110):
ret, frame = cap.read()

# Display the first frame (optional)
cv2_imshow(frame)


# Save the first frame (optional)
cv2.imwrite('cf.jpg', frame)

# Release the video capture object
cap.release()
